Evaluating the Finetuned Model

In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec spacy
!python -m spacy download en_core_web_trf

## Setup

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

CHECKPOINT_DIR = "asr_finetune/checkpoint"
processor = AutoProcessor.from_pretrained(CHECKPOINT_DIR)
model = AutoModelForSpeechSeq2Seq.from_pretrained(CHECKPOINT_DIR, device_map="auto")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

## Test Data Setup

In [ ]:
import torchaudio
from IPython.display import Audio, display




dataset_paths = [("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test", "voxpopuli"),
                 ("/training-1/asr/kaldi_data/cv19/en/test", "commonvoice"),
                 ("/training-1/asr/kaldi_data/singleword/en/test", "commonvoice_singlewords"),
                 ("/training-1/asr/kaldi_data/mls/en/test", "librispeech")

                 
                ]# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en/test /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en/test /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test *_de *_fr *_es
datasets = {}
for dataset_path, dataset_title in dataset_paths:
    dataset = []
    wav_scp_dict = {}
    with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
        for line in rf:
            id, wav_path = line.strip().split("\t")
            wav_scp_dict[id] = wav_path
    
    text_dict = {}
    with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
        for line in rf:
            id, text = line.strip().split("\t")
            text_dict[id] = text
    
    for id, wav_path in wav_scp_dict.items():
        text = text_dict[id]
        dataset.append({"audio" : wav_path, "text" : text})
    datasets[dataset_title] = dataset

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
'''
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))
'''

## Transcription Function

In [ ]:
def transcribe(wav, prompt=None, hotwords=None):
    """Granite Speech: chat-template prompt with <|audio|>; keywords appended for biasing."""
    if prompt is None:
        prompt = "can you transcribe the speech into a written format?"
    if hotwords:
        prompt = f"{prompt} Keywords: {hotwords}"
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{prompt}"},
    ]
    text = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text, wav, device=model.device, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1)
    return processor.tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                                      skip_special_tokens=True).strip()

## Text Normalization, Imports

In [ ]:
import jiwer
import re
import string

def normalize(text):
    text = text.translate(str.maketrans("", "", string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

## Run Inference on the Test Set

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_trf")
context_list = [False, True]

for c in context_list:
    for dataset_title, dataset_entries in datasets.items():
        eval_rows = []
        for sample in tqdm(dataset_entries[:50]):
            entities = None
            if c:
                doc = nlp(sample["text"])
                entities = [ent.text for ent in doc.ents]
                entities = ", ".join(entities)
            model_output = None
            if entities:
                model_output = transcribe(load_audio(sample["audio"]), hotwords=entities)
            else:
                model_output = transcribe(load_audio(sample["audio"]))
            wer = jiwer.wer(sample["text"], model_output)
            cer = jiwer.cer(sample["text"], model_output)
            eval_rows.append({"audio_path": sample["audio"],
                              "label": normalize(sample["text"]),
                              "prediction": normalize(model_output),
                              "wer" : wer,
                              "cer": cer
                             }
                         )
        output_statistics = []
        results = pd.DataFrame(eval_rows)
        results.head()
        output_statistics.append("Mean WER:", results["wer"].mean())
        output_statistics.append("Mean CER:", results["cer"].mean())
        results = results.sort_values("wer", ascending=False)
        output_name = "granite_finetuned_" + dataset_title
        if c:
            output_name = f"{output_name}_with_context"
        else:
            output_name = f"{output_name}_without_context"
        results.to_csv(f"{output_name}.tsv", sep="\t", index=False)
        with open(f"{output_name}_statistics.txt", "w", encoding="UTF-8") as wf:
            for line in output_statistics:
                wf.write(line+"\n")
            print("="*50)
            print(dataset_title)
            print(output_name)
            print("\n".join(output_statistics))
            print("="*50)

## Corpus-Level Results

In [ ]:
print("Mean WER:", results["wer"].mean())
print("Mean CER:", results["cer"].mean())
results[["wer", "cer"]].describe()
results = results.sort_values("wer", ascending=False)
results.to_csv("results_with_context.tsv", sep="\t", index=False)

## Worst Samples

In [ ]:
results.sort_values("wer", ascending=False).head(10)

In [ ]:
from IPython.display import Audio, display

for _, row in results.sort_values("wer", ascending=False).head(10).iterrows():
    print("=" * 70)
    print("AUDIO  :", row["audio_path"])
    print("LABEL    :", row["label"])
    print("PRED    :", row["prediction"])
    print(f"WER {row["wer"]:.2%} | CER {row["cer"]:.2%}")
    display(Audio(load_audio(row["audio_path"]), rate=16000))